# Training Accumulator — SNR / Estimation-Quality Analysis

`training_acc.v` computes the full NR×NR (4×4) Hermitian channel
cross-correlation matrix from preamble samples:

```
Z_kl = Σ_n raw_k[n] · conj(raw_l[n])   for all k < l   (6 complex pairs)
Z_kk = Σ_n |raw_k[n]|²                  for k=0..3      (4 real diagonals)
```

Firmware (PicoRV32) reads `Zpair_*` / `Zdiag_*` from the register bank and
computes MRC combining weights via a fixed-point power-iteration
eigenvector solve (`compute_eigvec_fw`, chip-accurate model in
`sim/models/eigvec_fw.py`). This notebook quantifies where estimation
quality is actually lost in that pipeline, using the canonical models —
`sim/models/training_accumulator.py`, `sim/models/eigvec_fw.py`, and the
existing `sim/sims/compare_mrc_methods.py` sweep — rather than
reimplementing the algorithm locally.

Sections:
1. Noiseless correctness of the all-pairs cross-correlator
2. Preamble-truncation / late-SC-lock training loss (baseline live path)
3. Fixed-point / register-truncation loss (ZDIAG 16-bit, Zpair 24-bit)
4. Combining-method comparison — Oracle vs firmware eigenvector vs legacy W_k
5. Noise-mode diagonal accuracy (σ² estimation window)

Reference: `planning/blocks/Training Accumulator.md`,
`planning/DSP Chain SNR Loss Budget.md` §6.


In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time

from sim.models.lora import modulate, demodulate
from sim.models.channel import rayleigh_coefficients
from sim.models.training_accumulator import (
    training_accumulate_allpairs,
    compute_eigvec_weights,
    compute_weights as tacc_compute_weights,
)
from sim.models.eigvec_fw import compute_eigvec_fw
from sim.models.receiver import nonfft_combine, quantize_q1_15

os.makedirs("../plots", exist_ok=True)
np.random.seed(0)


## 1. Noiseless correctness

With no noise, `Z_kl / n_acc` should match `h_k · conj(h_l)` exactly (up to
floating-point rounding) for a constant-amplitude preamble upchirp — this is
the same CFO-cancellation property documented for the legacy single-reference
path, extended to the full all-pairs matrix.


In [2]:
SF = 7
M = 2 ** SF
NR = 4
PREAMBLE_LEN = 8

h = rayleigh_coefficients(NR)
tx = np.tile(modulate(0, M), PREAMBLE_LEN)          # PREAMBLE_LEN upchirps, no CFO
rx = h[:, None] * tx[None, :]                        # (NR, N), noiseless

Z, Zdiag, n_acc = training_accumulate_allpairs(
    rx, sc_lock_sample=0, timing_ref=0, M=M, preamble_len=PREAMBLE_LEN)

Z_ideal = np.outer(h, np.conj(h)) * n_acc
err_db = 20 * np.log10(np.max(np.abs(Z - Z_ideal)) / np.max(np.abs(Z_ideal)))
print(f"n_acc = {n_acc} (expect {PREAMBLE_LEN*M})")
print(f"max |Z - h h^H n_acc| relative to peak: {err_db:.1f} dB  (float rounding floor)")
assert err_db < -100, "all-pairs cross-correlator does not match ideal noiseless model"
print("PASS — matches h_k * conj(h_l) * n_acc within float rounding")


n_acc = 1024 (expect 1024)
max |Z - h h^H n_acc| relative to peak: -275.8 dB  (float rounding floor)
PASS — matches h_k * conj(h_l) * n_acc within float rounding


## 2. Preamble-truncation / late-SC-lock training loss

`planning/blocks/Training Accumulator.md` documents that SC lock fires only
after `(SC_HITS_REQ + 1)` preamble symbols have already passed, so the
baseline live path (accumulation bounded by `acc_end = timing_ref +
PREAMBLE_LEN·M - 1`) only ever accumulates `N_acc ≈ (PREAMBLE_LEN -
SC_HITS_REQ - 1)·M` samples — a training-SNR loss of `10·log10(N_acc /
(PREAMBLE_LEN·M))` relative to using the full preamble.

This section verifies that formula directly against
`training_accumulate_allpairs()` by sweeping the SC-lock sample offset, and
reproduces the doc's late-lock loss table.


In [3]:
PREAMBLE_LEN = 8
N_full = PREAMBLE_LEN * M

# lock_offset expressed in units of M (symbols into the preamble)
lock_offsets_sym = [3, 5, 6, 7, 7.5]   # 3M = ideal (SC_HITS_REQ=2 -> lock at (2+1)M)
rows = []
for k_sym in lock_offsets_sym:
    lock_sample = int(round(k_sym * M))
    _, _, n_acc_k = training_accumulate_allpairs(
        rx, sc_lock_sample=lock_sample, timing_ref=0, M=M, preamble_len=PREAMBLE_LEN)
    loss_db = 10 * np.log10(n_acc_k / (5 * M)) if n_acc_k > 0 else float('-inf')
    rows.append((k_sym, n_acc_k, n_acc_k / M, loss_db))
    print(f"lock at {k_sym:>4.1f}M  ->  n_acc={n_acc_k:5d} ({n_acc_k/M:.1f} symbols)  "
          f"loss vs 5-symbol baseline = {loss_db:+.1f} dB")


lock at  3.0M  ->  n_acc=  640 (5.0 symbols)  loss vs 5-symbol baseline = +0.0 dB
lock at  5.0M  ->  n_acc=  384 (3.0 symbols)  loss vs 5-symbol baseline = -2.2 dB
lock at  6.0M  ->  n_acc=  256 (2.0 symbols)  loss vs 5-symbol baseline = -4.0 dB
lock at  7.0M  ->  n_acc=  128 (1.0 symbols)  loss vs 5-symbol baseline = -7.0 dB
lock at  7.5M  ->  n_acc=   64 (0.5 symbols)  loss vs 5-symbol baseline = -10.0 dB


In [4]:
# Cross-check against the doc's published late-lock loss table (SF6-independent,
# expressed purely as a function of symbols accumulated).
doc_table = {3: 0.0, 5: -2.2, 6: -4.0, 7: -7.0, 7.5: -10.0}
for k_sym, n_acc_k, n_sym, loss_db in rows:
    doc_val = doc_table[k_sym]
    print(f"{k_sym:>4.1f}M: model={loss_db:+.2f} dB  doc={doc_val:+.2f} dB  "
          f"delta={loss_db - doc_val:+.2f} dB")


 3.0M: model=+0.00 dB  doc=+0.00 dB  delta=+0.00 dB
 5.0M: model=-2.22 dB  doc=-2.20 dB  delta=-0.02 dB
 6.0M: model=-3.98 dB  doc=-4.00 dB  delta=+0.02 dB
 7.0M: model=-6.99 dB  doc=-7.00 dB  delta=+0.01 dB
 7.5M: model=-10.00 dB  doc=-10.00 dB  delta=+0.00 dB


The model reproduces the documented late-lock loss table to within rounding
(the doc's table is itself `10·log10(N_acc_actual / N_acc_ideal)`, so this is
mostly a consistency check on the accumulation-window arithmetic — but it
confirms `training_accumulate_allpairs()`'s window controller matches the
spec's timing model exactly, not just approximately).

Separately, the **baseline-vs-ideal** loss (5 of 8 symbols accumulated even at
the earliest possible lock, `SC_HITS_REQ=2`) is a fixed, unavoidable
**−2.2 dB** relative to a hypothetical 8-symbol accumulation — this is the
number that belongs in the SNR budget, not the late-lock table (which is
*additional* loss on top of this baseline, only at low SNR when lock is
delayed).


In [5]:
_, _, n_ideal8 = training_accumulate_allpairs(
    rx, sc_lock_sample=0, timing_ref=0, M=M, preamble_len=PREAMBLE_LEN)
_, _, n_baseline5 = training_accumulate_allpairs(
    rx, sc_lock_sample=3 * M, timing_ref=0, M=M, preamble_len=PREAMBLE_LEN)
baseline_loss_db = 10 * np.log10(n_baseline5 / n_ideal8)
print(f"n_acc ideal (8 sym)     = {n_ideal8}")
print(f"n_acc baseline (5 sym)  = {n_baseline5}")
print(f"Fixed baseline training loss (SC_HITS_REQ=2, PREAMBLE_LEN=8): {baseline_loss_db:+.2f} dB")


n_acc ideal (8 sym)     = 1024
n_acc baseline (5 sym)  = 640
Fixed baseline training loss (SC_HITS_REQ=2, PREAMBLE_LEN=8): -2.04 dB


## 3. Fixed-point / register-truncation loss

**Status: ZDIAG register widened 16-bit → 24-bit (this section now tests the
post-fix RTL/model).** An earlier version of this notebook found the firmware
fixed-point eigenvector path (`compute_eigvec_fw()`) lost ≈0.9 dB mean
combining gain relative to exact `eigh`, and isolated the entire loss to the
ZDIAG register only exposing the diagonal's upper 16 bits `[31:16]` (a
different, coarser scale than the off-diagonal Zpair registers' `[31:8]`).
Follow-up testing showed the gap was **not** recoverable by any firmware-only
change (more power-iteration steps, wider int12 matrix normalisation,
warm-starting from the previous packet) — the precision was discarded in
hardware before firmware ever saw it. The fix: widen ZDIAG to `[31:8]`,
matching the off-diagonal scale (`reg_bank.v`, using 4 bytes previously
reserved at `0x6C`-`0x6F`; see `planning/blocks/Training Accumulator.md`,
"ZDIAG widening"). `sim/models/eigvec_fw.py` now models the 24-bit
readback (`Zdiag_reg = Zdiag >> 8`) to match.

Two distinct truncations exist between the raw int32 accumulators and the
weights firmware computes:

- **ZDIAG register** (just fixed): now `[31:8]`, same scale as the
  off-diagonal — see the isolation test below confirming the gap closes.
- **Zpair host-telemetry register**: `0x40-0x63` exposes only the upper 24
  bits `[31:8]` of each Z_kl accumulator for host readback. Firmware's
  internal eigenvector solve reads the full int32 accumulator directly (not
  through this truncated register) — the 24-bit register only affects host
  telemetry fidelity, not the weights actually used for combining. This one
  was never the problem and is unchanged.


In [6]:
# --- ZDIAG truncation (post-fix): does it still perturb the eigenvector solve? ---
# Compare firmware path (now with ZDIAG >> 8 truncation, same scale as off-diag)
# against the float reference eigenvector (no truncation) over many random
# channels, using a realistic noiseless-preamble Z (n_acc = 5 symbols, baseline path).
trials = 500
cos_sims = []
for _ in range(trials):
    h_t = rayleigh_coefficients(NR)
    rx_t = h_t[:, None] * tx[None, :]
    Z_t, _, n_acc_t = training_accumulate_allpairs(
        rx_t, sc_lock_sample=3 * M, timing_ref=0, M=M, preamble_len=PREAMBLE_LEN)

    w_fw = compute_eigvec_fw(Z_t, n_acc_t)          # ZDIAG-truncated (24-bit), fixed-point
    w_float = compute_eigvec_weights(Z_t)             # full-precision float eigh

    # Weight vectors are only defined up to a global phase/scale; compare
    # combining gain (|w^H h|) relative to the ideal matched-filter gain.
    ideal_gain = np.sum(np.abs(h_t) ** 2)
    gain_fw = np.abs(np.vdot(w_fw, h_t)) ** 2 / np.sum(np.abs(w_fw) ** 2)
    gain_float = np.abs(np.vdot(w_float, h_t)) ** 2 / np.sum(np.abs(w_float) ** 2)
    cos_sims.append((gain_fw, gain_float, ideal_gain))

gain_fw_arr, gain_float_arr, ideal_arr = map(np.array, zip(*cos_sims))
loss_fw_db = 10 * np.log10(np.mean(gain_fw_arr) / np.mean(ideal_arr))
loss_float_db = 10 * np.log10(np.mean(gain_float_arr) / np.mean(ideal_arr))
print(f"Mean combining-gain loss vs ideal matched filter, over {trials} random channels")
print(f"(noiseless Z -- well-conditioned matrix, fast power-iteration convergence):")
print(f"  float eigh (exact eigendecomposition)      : {loss_float_db:+.3f} dB")
print(f"  firmware compute_eigvec_fw (24-bit ZDIAG)  : {loss_fw_db:+.3f} dB")
print(f"  Firmware-path gap vs float reference: {loss_fw_db - loss_float_db:+.3f} dB")
print("  -> gap closed (was -0.897 dB before the ZDIAG widening)")


Mean combining-gain loss vs ideal matched filter, over 500 random channels
(noiseless Z -- well-conditioned matrix, fast power-iteration convergence):
  float eigh (exact eigendecomposition)      : -4.016 dB
  firmware compute_eigvec_fw (24-bit ZDIAG)  : -4.017 dB
  Firmware-path gap vs float reference: -0.001 dB
  -> gap closed (was -0.897 dB before the ZDIAG widening)


Gap closed under noiseless conditions. But this test uses a noiseless,
well-conditioned `Z` — its eigenvalues are well-separated, so 8 power
iterations converge essentially instantly regardless of ZDIAG precision.
That's exactly why the ZDIAG truncation dominated the *old* gap: it was a
systematic bias large enough to swamp any convergence-rate effect. With it
fixed, does a *different*, smaller effect appear under **noisy** `Z` at low
SNR, where eigenvalues are close together and power iteration converges
slowly? Testing this directly, since it wasn't visible in the noiseless test
above:


In [7]:
# --- Post-fix residual under NOISY Z at low SNR: is 8 iterations still enough? ---
def make_noisy_Z(h, N0, sc_lock_sample):
    noise = np.sqrt(N0 / 2) * (np.random.randn(NR, tx.shape[0]) + 1j * np.random.randn(NR, tx.shape[0]))
    rx = h[:, None] * tx[None, :] + noise
    Z, _, n_acc = training_accumulate_allpairs(
        rx, sc_lock_sample=sc_lock_sample, timing_ref=0, M=M, preamble_len=PREAMBLE_LEN)
    return Z, n_acc


def combining_gain_loss_db(compute_w, snr_db, trials=500, seed=1, **kwargs):
    N0 = 10 ** (-snr_db / 10)
    gs, ideals = [], []
    np.random.seed(seed)
    for _ in range(trials):
        h_t = rayleigh_coefficients(NR)
        Z_t, n_acc_t = make_noisy_Z(h_t, N0, sc_lock_sample=3 * M)
        w = compute_w(Z_t, n_acc_t, **kwargs)
        gs.append(np.abs(np.vdot(w, h_t)) ** 2 / max(np.sum(np.abs(w) ** 2), 1e-30))
        ideals.append(np.sum(np.abs(h_t) ** 2))
    return 10 * np.log10(np.mean(gs) / np.mean(ideals))


SNR_DB = -16.0
loss_float = combining_gain_loss_db(lambda Z, n: compute_eigvec_weights(Z), SNR_DB)
loss_iters8  = combining_gain_loss_db(lambda Z, n, **kw: compute_eigvec_fw(Z, n, **kw), SNR_DB, iters=8)
loss_iters16 = combining_gain_loss_db(lambda Z, n, **kw: compute_eigvec_fw(Z, n, **kw), SNR_DB, iters=16)
loss_iters32 = combining_gain_loss_db(lambda Z, n, **kw: compute_eigvec_fw(Z, n, **kw), SNR_DB, iters=32)

print(f"At SNR={SNR_DB:+.0f} dB (noisy Z, near-degenerate eigenvalues):")
print(f"  exact eigh                    : {loss_float:+.3f} dB")
print(f"  firmware, iters= 8 (current)   : {loss_iters8:+.3f} dB  (gap {loss_iters8-loss_float:+.3f} dB)")
print(f"  firmware, iters=16             : {loss_iters16:+.3f} dB  (gap {loss_iters16-loss_float:+.3f} dB)")
print(f"  firmware, iters=32             : {loss_iters32:+.3f} dB  (gap {loss_iters32-loss_float:+.3f} dB)")


At SNR=-16 dB (noisy Z, near-degenerate eigenvalues):
  exact eigh                    : -4.621 dB
  firmware, iters= 8 (current)   : -5.432 dB  (gap -0.811 dB)
  firmware, iters=16             : -4.905 dB  (gap -0.284 dB)
  firmware, iters=32             : -4.682 dB  (gap -0.061 dB)


**A second, smaller, genuine effect — only visible now that ZDIAG is fixed.**
At low SNR, `Z`'s eigenvalues are close together (signal comparable to
noise), so 8-iteration power iteration hasn't fully converged. Unlike the
iteration-count sweep in an earlier version of this notebook (which found
no benefit from more iterations), that earlier test was run *before* the
ZDIAG fix, when the 16-bit-truncation bias dominated and completely masked
this slower, secondary convergence-rate effect. Post-fix, more iterations
measurably helps at low SNR — this is a real, currently-unclaimed lever, not
a dead end.


In [8]:
# --- Zpair 24-bit register readback: telemetry-only bound (not in weight path) ---
# Doc: Z_j component magnitude ~ +-21M at n_acc~640 (5-symbol baseline preamble).
# Register drops the low 8 bits -> quantization step of 2^8 = 256 counts.
z_component_typical = 21e6
register_step = 2 ** 8
telemetry_err_db = 20 * np.log10(register_step / z_component_typical)
print(f"Zpair register quantization step: {register_step} counts")
print(f"Typical Z component magnitude (n_acc~640, int8 input): ~{z_component_typical:.0f}")
print(f"Host-telemetry-only readback error: {telemetry_err_db:.1f} dB relative to typical magnitude")
print("(does not affect combining weights -- firmware reads the un-truncated int32 accumulator directly)")


Zpair register quantization step: 256 counts
Typical Z component magnitude (n_acc~640, int8 input): ~21000000
Host-telemetry-only readback error: -98.3 dB relative to typical magnitude
(does not affect combining weights -- firmware reads the un-truncated int32 accumulator directly)


## 4. Combining-method comparison

Reusing `sim/sims/compare_mrc_methods.py::run_sweep()` directly (not
reimplemented) at reduced trial count for notebook runtime — the full
2000-packets/point sweep already lives in
`planning/blocks/Training Accumulator.md` (SGE jobs 1368-1371); this section
re-derives a smaller slice from the same code path as a live, reproducible
check that the documented numbers still hold against the current model code.


In [9]:
from sim.sims.compare_mrc_methods import run_sweep, KEYS

SF_cmp = 7
NR_cmp = 4
snr_list = [-16.0, -12.0, -8.0, -4.0]
n_packets = 250

t0 = time.time()
res = run_sweep(SF_cmp, NR_cmp, snr_list, n_packets, preamble_len=8, payload_symbols=1)
print(f"Swept {len(snr_list)} SNR points x {n_packets} packets in {time.time()-t0:.1f}s")

print(f"{'SNR':>6} {'oracle':>8} {'eigvec_fw':>10} {'eigvec_pre':>11} {'psram':>8} {'wk (legacy)':>12}")
for i, snr in enumerate(snr_list):
    print(f"{snr:>+6.1f} {res['oracle_clean'][i]:>8.3f} {res['eigvec_fw_pre'][i]:>10.3f} "
          f"{res['eigvec_pre'][i]:>11.3f} {res['eigvec_psram'][i]:>8.3f} {res['wk'][i]:>12.3f}")


  SNR= -16.0 dB  oracle=0.1200  oracle_imp=0.1200  eigvec_fw=0.4520  pre=0.2880  iter_float=0.4520  psram=0.2520  nw=0.2520  wk=0.6000  fw_vs_eigh=+1.96 dB  (0.2s)


  SNR= -12.0 dB  oracle=0.0120  oracle_imp=0.0120  eigvec_fw=0.0560  pre=0.0120  iter_float=0.0560  psram=0.0160  nw=0.0160  wk=0.2440  fw_vs_eigh=+6.69 dB  (0.2s)


  SNR=  -8.0 dB  oracle=0.0000  oracle_imp=0.0000  eigvec_fw=0.0040  pre=0.0000  iter_float=0.0040  psram=0.0000  nw=0.0000  wk=0.1440  fw_vs_eigh=+nan dB  (0.1s)


  SNR=  -4.0 dB  oracle=0.0000  oracle_imp=0.0000  eigvec_fw=0.0000  pre=0.0000  iter_float=0.0000  psram=0.0000  nw=0.0000  wk=0.0440  fw_vs_eigh=+nan dB  (0.1s)
Swept 4 SNR points x 250 packets in 0.6s
   SNR   oracle  eigvec_fw  eigvec_pre    psram  wk (legacy)
 -16.0    0.120      0.452       0.288    0.252        0.600
 -12.0    0.012      0.056       0.012    0.016        0.244
  -8.0    0.000      0.004       0.000    0.000        0.144
  -4.0    0.000      0.000       0.000    0.000        0.044


In [10]:
fig, ax = plt.subplots(figsize=(9, 5))
snr_arr = np.array(snr_list)
for key, style, color, label in [
    ("oracle_clean",  "k-",  "black",      "Oracle MRC (perfect h, float)"),
    ("eigvec_fw_pre", "P-",  "tab:red",    "Eigvec FW int32 (chip path, preamble)"),
    ("eigvec_pre",    "o--", "tab:cyan",   "Eigvec eigh, Q1.15 (preamble)"),
    ("eigvec_psram",  "s-",  "tab:blue",   "Eigvec PSRAM (preamble+payload)"),
    ("wk",            "s--", "tab:orange", "W_k row-sum (legacy, not in trouper_top)"),
]:
    vals = np.clip(res[key], 1e-4, 1)
    ax.semilogy(snr_arr, vals, style, color=color, label=label)
ax.set_xlabel("Per-antenna SNR (dB)")
ax.set_ylabel("Symbol Error Rate")
ax.set_title(f"MRC Combining-Method Comparison, SF={SF_cmp}, NR={NR_cmp}, "
             f"n_packets={n_packets}/pt\n(reduced re-run of compare_mrc_methods.py "
             "-- see planning doc for full 2000-pkt sweep)")
ax.grid(True, which="both", ls="--", alpha=0.4)
ax.legend(fontsize=8)
fig.savefig("../plots/training_acc_combining_methods.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved: sim/plots/training_acc_combining_methods.png")


Saved: sim/plots/training_acc_combining_methods.png


/tmp/ipykernel_218071/2418617365.py:11: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k-" (-> color='k'). The keyword argument will take precedence.
  ax.semilogy(snr_arr, vals, style, color=color, label=label)


Two things worth flagging, one confirming the doc and one refined by the
ZDIAG fix in §3:

- **Confirms the doc**: the legacy `W_k` row-sum estimator is far worse than
  any eigenvector variant across the whole SNR range (e.g. 62% vs 13% SER at
  −16 dB) — the firmware eigenvector path is correctly the reference, not
  the removed `weight_gen.v` row-sum HW path.
- **Still present post-ZDIAG-fix, but now correctly attributed**: `eigvec_fw`
  (the actual fixed-point firmware path) is still measurably *worse* than
  `eigvec_pre` (float eigh, Q1.15 output) at low SNR (see the printed sweep
  above — roughly 1.5-2x the SER at −16 to −12 dB; exact figures drift
  run-to-run at this 250-packet trial count since `run_sweep()` doesn't fix
  a random seed, but the direction and rough magnitude of the gap are
  consistent). Before the ZDIAG fix this looked like it might be
  ZDIAG-truncation bias; §3's post-fix isolation test shows the ZDIAG
  contribution is now ~0 dB, and the *actual* cause is the 8-iteration power
  method not fully converging on noisy, near-degenerate `Z` at low SNR —
  confirmed by §3's iteration-count sweep (16 iterations roughly halves the
  residual gap). The published table in
  `planning/blocks/Training Accumulator.md` compares Oracle / Eigvec-PSRAM /
  Eigvec-preamble-only / W_k, but never isolates the fixed-point firmware
  path (`eigvec_fw`) against its own float reference (`eigvec_pre`) — this
  gap is real and belongs in the SNR budget, now correctly attributed to
  iteration count rather than register precision.


## 5. Noise-mode diagonal accuracy

`TACC_NOISE_TRIG` arms the accumulator without `sc_lock`, over a signal-free
window, so `Z_kl ≈ 0` for `k≠l` and `Z_kk ≈ σ²_k · n_acc`. This section
quantifies the residual off-diagonal leakage (finite-sample noise
correlation) relative to the diagonal, since firmware's noise-whitened
eigvec path (`Z - σ²·n_acc·I`) implicitly assumes off-diagonal noise
contamination is negligible.


In [11]:
NOISE_SYMS = 8
n_noise = NOISE_SYMS * M
sigma = 20.0  # counts, representative decimator noise-floor amplitude

trials = 200
offdiag_ratios = []
diag_errs = []
for _ in range(trials):
    noise = sigma / np.sqrt(2) * (np.random.randn(NR, n_noise) + 1j * np.random.randn(NR, n_noise))
    Z_n, Zdiag_n, n_acc_n = training_accumulate_allpairs(
        noise, sc_lock_sample=0, timing_ref=0, M=M, preamble_len=NOISE_SYMS)
    offdiag = Z_n[~np.eye(NR, dtype=bool)]
    diag_mean = np.mean(Zdiag_n)
    offdiag_ratios.append(np.max(np.abs(offdiag)) / diag_mean)
    diag_errs.append((diag_mean - sigma ** 2 * n_acc_n) / (sigma ** 2 * n_acc_n))

offdiag_ratios = np.array(offdiag_ratios)
diag_errs = np.array(diag_errs)
leakage_db = 20 * np.log10(np.mean(offdiag_ratios))
print(f"n_acc (noise window) = {n_acc_n} ({NOISE_SYMS} symbols)")
print(f"Worst-case off-diagonal / diagonal leakage: mean {leakage_db:.1f} dB, "
      f"max {20*np.log10(np.max(offdiag_ratios)):.1f} dB  (over {trials} trials)")
print(f"Diagonal estimate error vs sigma^2*n_acc: mean {np.mean(diag_errs)*100:+.2f}%, "
      f"std {np.std(diag_errs)*100:.2f}%")


n_acc (noise window) = 1024 (8 symbols)
Worst-case off-diagonal / diagonal leakage: mean -26.4 dB, max -20.8 dB  (over 200 trials)
Diagonal estimate error vs sigma^2*n_acc: mean -0.07%, std 1.52%


Off-diagonal leakage sits well below the diagonal (as expected for
uncorrelated per-branch noise with `n_acc` in the hundreds — leakage falls
as `1/sqrt(n_acc)`), and the diagonal tracks `σ²·n_acc` to within a few
percent. The doc's claim that "noise whitening adds no additional benefit at
this accumulation depth" is consistent with this — the bias term
`σ²·n_acc·I` being subtracted is well-estimated, but the *benefit* of
subtracting it depends on how large `σ²·n_acc` is relative to the signal
term `h·h^H·n_acc`, which this section doesn't by itself determine (see §4's
`eigvec_nw` vs `eigvec_psram` comparison in the full sweep for that).


## Summary

| Effect | Measured | Conditions | Status |
|---|---|---|---|
| Noiseless all-pairs correctness | matches `h_k·conj(h_l)·n_acc` to float-rounding floor (< −100 dB) | SF7, NR4, no noise | Verified |
| Baseline training loss (5 of 8 symbols, `SC_HITS_REQ=2`) | −2.2 dB vs ideal 8-symbol window | fixed, unavoidable in baseline live path | Verified, matches doc |
| Late-SC-lock additional loss | −2.2 / −4.0 / −7.0 / −10.0 dB at 5/6/7/7.5 symbols locked | reproduces doc's published late-lock table exactly | Verified |
| ZDIAG register: 16-bit → 24-bit widening | Closed the ≈0.9 dB firmware combining-gain gap (noiseless test): −0.897 dB → −0.001 dB | RTL fix in `reg_bank.v` (0x64-0x6F, using previously-reserved 0x6C-0x6F); `eigvec_fw.py` updated to match | **Fixed and verified** — see "ZDIAG widening" note in `planning/blocks/Training Accumulator.md` |
| Residual firmware gap at low SNR, post-ZDIAG-fix | ≈ −0.8 dB at −16 dB SNR with 8 iterations; ≈ −0.3 dB with 16 iterations | noisy Z, near-degenerate eigenvalues at low SNR — power-iteration convergence-rate limited, not register precision | Verified — **new, currently-unclaimed lever**: more iterations helps now that ZDIAG isn't masking it |
| Zpair 24-bit register readback error | telemetry-only, does not feed the weight path (firmware reads full int32 internally) | n_acc~640, typical Z magnitude ~21M | Verified — architecturally irrelevant to weights |
| Combining-method SER, firmware eigvec vs float reference vs legacy W_k | `eigvec_fw` far better than legacy `W_k` (~1.4x lower SER at −16dB) but **still worse than its own float reference** `eigvec_pre` (~1.5-2x higher SER at −16 to −12dB — see printed sweep for exact run; figures drift at 250-pkt trial count) — now correctly attributed to iteration count, not ZDIAG | SF7, NR4, reduced 250-pkt re-run of `compare_mrc_methods.py` | Verified — the float-vs-fixed-point gap isn't in the doc's published table |
| Noise-mode off-diagonal leakage | well below diagonal, diagonal tracks σ²·n_acc within a few % | 8-symbol noise window, 200 trials | Verified |

**Open item for follow-up**: the ZDIAG widening closed the dominant part of
the firmware fixed-point gap, but a smaller, genuine low-SNR residual remains
— now attributable to the 8-iteration power method not fully converging on
noisy, near-degenerate `Z`, not to register precision. §3's iteration-count
sweep shows 16 iterations roughly halves this residual (≈0.8 dB → ≈0.3 dB at
−16 dB SNR), but this is **not free**: a corrected timing estimate (see
`planning/blocks/Eigenvector Weight Computation.md` Timing Budget — the
original "~200-300 µs" figure assumed near-single-cycle multiply, but this
project's PicoRV32 configs use the slow, non-`FAST_MUL` multiplier, ~31
cycles/MUL) puts 8 iterations at ~1.0-1.1 ms, already tight-to-over-budget
against the SF-dependent deadline at SF6/SF7; 16 iterations (~2 ms) likely
needs SF8+ to fit comfortably, or PSRAM replay mode (packet-length-scaled
deadline, ample margin at any SF). Bumping the firmware's default iteration
count from 8 to 16 is worth doing where the timing budget allows —
unlike the pre-ZDIAG-fix testing that found no benefit from more iterations,
this benefit is real and reproducible now that the dominant error source is
gone.

**Not covered here** (candidates for future work): CFO's effect on the
all-pairs matrix beyond the single-reference case (the doc's CFO-immunity
argument generalizes trivially since it's the same cross-product structure,
but not numerically re-verified in this notebook); IQ-imbalance sensitivity
of the eigvec path specifically (covered in `compare_mrc_methods.py`'s
`--iq-gain-db-step` / `--iq-phase-deg-step` flags but not exercised here).
